# 02 - The pricing core

Milestone 2 of `vol-strategy-backtester` prices every quote: a forward per expiry from put-call parity, an implied vol per out-of-the-money quote, and a smile per expiry. SPY options are American, so first the early-exercise premium is measured on a binomial lattice and taken off each mid. This notebook copies five modules out of `src/volbt/pricing/` and runs them on the ADRs' own worked numbers. It needs no market data and no `volbt`.

This notebook is generated by `tools/build_notebooks.py`. Do not edit it by hand; edit `src/` and rebuild. The measured results of the full build are in `docs/milestones/02-pricing-core.md`.

## Black-76 on the forward

Black-Scholes written in the forward F instead of the spot: the dividend is inside F, so no dividend yield is needed. The greeks come in ADR-0006's USD units. Copied from `src/volbt/pricing/black76.py`.

In [ ]:
import numpy as np
from scipy.special import ndtr

In [ ]:
# 1 / sqrt(2 pi), for the normal density n(x) = exp(-x^2 / 2) / sqrt(2 pi).
INV_SQRT_2PI = 1.0 / np.sqrt(2.0 * np.pi)
# ADR-0006 units: one SPY contract is 100 shares, gamma per 1% move, vega per 1 vol point, theta
# per calendar day.
CONTRACT_MULTIPLIER = 100
ONE_PERCENT = 0.01
ONE_VOL_POINT = 0.01
DAYS_PER_YEAR = 365

In [ ]:
def normal_pdf(x):
    """n(x), the standard normal density."""
    return INV_SQRT_2PI * np.exp(-0.5 * np.asarray(x, dtype=float) ** 2)

In [ ]:
def d1_d2(forward, strike, tau, sigma):
    """(d1, d2). Needs tau > 0 and sigma > 0: at expiry there is no distribution left to integrate."""
    forward, strike, tau, sigma = (np.asarray(x, dtype=float) for x in (forward, strike, tau, sigma))
    spread = sigma * np.sqrt(tau)          # the standard deviation of ln(F_T / F)
    d1 = (np.log(forward / strike) + 0.5 * spread**2) / spread
    return d1, d1 - spread

In [ ]:
def price(forward, strike, tau, rate, sigma, is_call):
    """The Black-76 price of a European call (is_call True) or put (False), per share."""
    d1, d2 = d1_d2(forward, strike, tau, sigma)
    df = np.exp(-np.asarray(rate, dtype=float) * np.asarray(tau, dtype=float))
    forward, strike = np.asarray(forward, dtype=float), np.asarray(strike, dtype=float)
    call = df * (forward * ndtr(d1) - strike * ndtr(d2))
    put = df * (strike * ndtr(-d2) - forward * ndtr(-d1))
    return np.where(is_call, call, put)

In [ ]:
def intrinsic(spot, strike, is_call):
    """max(S - K, 0) for a call, max(K - S, 0) for a put: the value at expiry (ADR-0008 §5)."""
    spot, strike = np.asarray(spot, dtype=float), np.asarray(strike, dtype=float)
    return np.where(is_call, np.maximum(spot - strike, 0.0), np.maximum(strike - spot, 0.0))

In [ ]:
def price_bounds(forward, strike, tau, rate, is_call):
    """(lower, upper): every European price lies strictly between them (ADR-0008 §7).

    A call is worth more than its discounted intrinsic value DF max(F - K, 0) and less than the
    discounted forward DF F; a put more than DF max(K - F, 0) and less than DF K. A price on or
    outside a bound implies no volatility at all, so the IV solver refuses it (IV_FAIL).
    """
    forward, strike = np.asarray(forward, dtype=float), np.asarray(strike, dtype=float)
    df = np.exp(-np.asarray(rate, dtype=float) * np.asarray(tau, dtype=float))
    lower = np.where(is_call, df * np.maximum(forward - strike, 0.0), df * np.maximum(strike - forward, 0.0))
    upper = np.where(is_call, df * forward, df * strike)
    return lower, upper

In [ ]:
def vega(forward, strike, tau, rate, sigma):
    """dV/dsigma per share, per 1.00 of vol (ADR-0006's vega_1vol_usd is this x 0.01 x 100 x qty)."""
    d1, _ = d1_d2(forward, strike, tau, sigma)
    df = np.exp(-np.asarray(rate, dtype=float) * np.asarray(tau, dtype=float))
    return df * np.asarray(forward, dtype=float) * normal_pdf(d1) * np.sqrt(np.asarray(tau, dtype=float))

In [ ]:
def greeks(spot, forward, strike, tau, rate, sigma, is_call):
    """{price, delta, gamma, vega, theta} per share (formulas in the module docstring).

    delta and gamma are with respect to the SPOT S, holding F/S fixed; theta is per year.
    """
    spot, forward, strike, tau, rate, sigma = (np.asarray(x, dtype=float)
                                               for x in (spot, forward, strike, tau, rate, sigma))
    d1, _ = d1_d2(forward, strike, tau, sigma)
    df = np.exp(-rate * tau)
    density = normal_pdf(d1)
    value = price(forward, strike, tau, rate, sigma, is_call)
    ratio = forward / spot
    return {
        "price": value,
        "delta": np.where(is_call, df * ratio * ndtr(d1), -df * ratio * ndtr(-d1)),
        "gamma": df * forward * density / (spot**2 * sigma * np.sqrt(tau)),
        "vega": df * forward * density * np.sqrt(tau),
        "theta": rate * value - df * forward * density * sigma / (2 * np.sqrt(tau)),
    }

In [ ]:
def greeks_usd(quantity, spot, forward, strike, tau, rate, sigma, is_call):
    """ADR-0006's units for `quantity` contracts (positive = long), as USD amounts.

        delta_shares    q 100 delta                      share-equivalent exposure
        delta_usd       delta_shares S                   P&L for a +100% move, to first order
        gamma_1pct_usd  q 100 gamma S^2 0.01             change in delta_usd for a +1% move in S
        vega_1vol_usd   q 100 vega 0.01                  P&L when IV rises by 1 vol point
        theta_1d_usd    q 100 theta / 365                P&L from one calendar day passing

    One long call at S = F = K = 400, sigma 0.20, 30 days, r = 0: $20,457.44, $2,781.94, $45.73
    and -$15.24 (ADR-0006 test 1).
    """
    per_share = greeks(spot, forward, strike, tau, rate, sigma, is_call)
    contracts = np.asarray(quantity, dtype=float) * CONTRACT_MULTIPLIER
    spot = np.asarray(spot, dtype=float)
    delta_shares = contracts * per_share["delta"]
    return {
        "delta_shares": delta_shares,
        "delta_usd": delta_shares * spot,
        "gamma_1pct_usd": contracts * per_share["gamma"] * spot**2 * ONE_PERCENT,
        "vega_1vol_usd": contracts * per_share["vega"] * ONE_VOL_POINT,
        "theta_1d_usd": contracts * per_share["theta"] / DAYS_PER_YEAR,
    }

In [ ]:
# ADR-0006 test 1: one long call, S = F = K = 400, sigma 0.20, 30 days, r = 0.
g = greeks_usd(1, 400, 400, 400, 30 / 365, 0.0, 0.20, True)
print({name: round(float(value), 2) for name, value in g.items()})
# Put-call parity: C - P = DF (F - K).
c, p = price(441.0, 430.0, 0.25, 0.05, 0.2, True), price(441.0, 430.0, 0.25, 0.05, 0.2, False)
print('C - P =', round(float(c - p), 6), ' DF (F - K) =', round(float(np.exp(-0.05 * 0.25) * 11.0), 6))

## The IV solver

Bounds first, then the OTM side (an ITM price is its bound plus the OTM price), Newton from the Brenner-Subrahmanyam guess to a tolerance relative to the price and to sigma, then bisection wherever Newton cannot be trusted, and "unresolved" where float64 cannot pin sigma. Copied from `src/volbt/pricing/iv.py`; the module imports two Black-76 functions under other names, so the next cell gives them those names.

In [ ]:
b76_price, b76_vega = price, vega

In [ ]:
import numpy as np

from scipy.special import ndtr

In [ ]:
PRICE_RTOL = 1e-10            # Newton has converged when |B - target| <= this x target...
NEWTON_SIGMA_RTOL = 1e-10     # ...and its next step |B - target| / vega is below this x sigma
SIGMA_RTOL = 1e-10            # bisection has converged when the bracket is narrower than this x sigma
RESOLVE_SIGMA_RTOL = 1e-6     # and float's rounding of B, over vega, must pin sigma to this x sigma
ROUNDING_ULPS = 8             # the rounding of a Black-76 price: this many epsilons of its two terms
NEWTON_MAX_ITER = 50
BISECTION_MAX_ITER = 100      # 5 / 2^100 is far below float precision in sigma: always enough
SIGMA_LO, SIGMA_HI = 1e-4, 5.0  # the search range, and Newton's bracket
GUESS_LO, GUESS_HI = 0.01, 3.0  # where the starting guess is clipped
VEGA_MIN = 1e-8

SOLVER_NONE, SOLVER_NEWTON, SOLVER_BISECTION = 0, 1, 2
SOLVER_NAMES = ("", "newton", "bisection")
(FAIL_NONE, FAIL_BELOW_BOUND, FAIL_ABOVE_BOUND, FAIL_NO_ROOT, FAIL_UNRESOLVED,
 FAIL_AMERICAN_BELOW_FLOOR, FAIL_AMERICAN_ABOVE_CAP, FAIL_AMERICAN_NO_CONVERGENCE, FAIL_AMERICAN_UNIDENTIFIED,
 FAIL_AMERICAN_NO_MID, FAIL_AMERICAN_EXPIRED, FAIL_AMERICAN_BAD_INPUT) = range(12)
FAIL_NAMES = ("", "below_lower_bound", "above_upper_bound", "no_root", "unresolved",
              "american_below_floor", "american_above_cap", "american_no_convergence", "american_unidentified",
              "american_no_mid", "american_expired", "american_bad_input")

In [ ]:
def price_rounding(forward, strike, tau, rate, sigma, is_call):
    """How far float64 can be off in a Black-76 price: ROUNDING_ULPS epsilons of its two terms.

    B = DF [F N(d1) - K N(d2)] (a call) is a difference of two terms; when they nearly cancel, as for
    a price close to its upper bound, the rounding is set by the terms, not by B. Near float64's
    smallest normal number the rounding is absolute, so a few of it are added.
    """
    d1, d2 = d1_d2(forward, strike, tau, sigma)
    df = np.exp(-np.asarray(rate, dtype=float) * np.asarray(tau, dtype=float))
    terms = np.where(is_call, forward * ndtr(d1) + strike * ndtr(d2), strike * ndtr(-d2) + forward * ndtr(-d1))
    # Below float64's smallest normal number (2.2e-308) precision is absolute, not relative.
    return ROUNDING_ULPS * (np.finfo(float).eps * df * terms + np.finfo(float).tiny)

In [ ]:
def brenner_subrahmanyam(price, forward, tau, rate):
    """The starting guess sqrt(2 pi / tau) V / (DF F), clipped to [0.01, 3]."""
    df = np.exp(-rate * tau)
    guess = np.sqrt(2 * np.pi / tau) * price / (df * forward)
    return np.clip(guess, GUESS_LO, GUESS_HI)

In [ ]:
def implied_vol(price, forward, strike, tau, rate, is_call):
    """Solve Black-76 for sigma, quote by quote. Arrays broadcast; every input needs tau > 0.

    Returns {"iv": float array (NaN where no IV), "solver": SOLVER_* codes, "iterations": ints,
    "fail": FAIL_* codes}. Never raises on a bad price: a price with no IV gets a fail code.
    """
    arrays = np.broadcast_arrays(*(np.asarray(x, dtype=float) for x in (price, forward, strike, tau, rate)),
                                 np.asarray(is_call, dtype=bool))
    price, forward, strike, tau, rate = (a.astype(float).ravel() for a in arrays[:5])
    is_call = arrays[5].ravel()
    n = price.size
    iv = np.full(n, np.nan)
    solver = np.zeros(n, dtype=np.int8)
    iterations = np.zeros(n, dtype=np.int16)
    fail = np.zeros(n, dtype=np.int8)

    # 1. Bounds. A NaN price fails both comparisons, so it is caught as "below"; so is a NaN input.
    with np.errstate(invalid="ignore"):
        lower, upper = price_bounds(forward, strike, tau, rate, is_call)
    fail[~(price > lower)] = FAIL_BELOW_BOUND
    fail[(price > lower) & ~(price < upper)] = FAIL_ABOVE_BOUND
    fail[(fail == FAIL_NONE) & ~(tau > 0)] = FAIL_NO_ROOT          # at expiry there is no sigma to find
    # 2. The OTM side: the time value is the OTM option's price at the same strike (put-call parity).
    otm_call = strike >= forward
    target = price - lower                      # > 0 wherever price > lower: IEEE subtraction of two
                                                # different finite floats is never 0
    # Inside the bounds but outside what sigma in [1e-4, 5] can produce: no root to find.
    todo = np.flatnonzero(fail == FAIL_NONE)
    args = (forward[todo], strike[todo], tau[todo], rate[todo])
    at_lo = b76_price(*args, SIGMA_LO, otm_call[todo])
    at_hi = b76_price(*args, SIGMA_HI, otm_call[todo])
    no_root = (target[todo] < at_lo) | (target[todo] > at_hi)
    fail[todo[no_root]] = FAIL_NO_ROOT
    todo = todo[~no_root]

    # 3. Newton from the Brenner-Subrahmanyam guess, on the time value.
    sigma = brenner_subrahmanyam(price[todo], forward[todo], tau[todo], rate[todo])
    active = np.arange(todo.size)               # positions in `todo` still iterating
    to_bisect = []
    for step in range(1, NEWTON_MAX_ITER + 1):
        if active.size == 0:
            break
        where = todo[active]
        model = b76_price(forward[where], strike[where], tau[where], rate[where], sigma[active], otm_call[where])
        error = model - target[where]
        slope = b76_vega(forward[where], strike[where], tau[where], rate[where], sigma[active])
        # Converged: the price matches to 1e-10 of itself AND the Newton step it would take next,
        # error / vega, is below 1e-10 of sigma (a flat price can match at the wrong sigma).
        done = (np.abs(error) <= PRICE_RTOL * target[where]) & (np.abs(error) <= NEWTON_SIGMA_RTOL * sigma[active] * slope)
        iv[where[done]] = sigma[active][done]
        solver[where[done]] = SOLVER_NEWTON
        iterations[where[done]] = step - 1
        keep = ~done
        active, error, slope = active[keep], error[keep], slope[keep]
        with np.errstate(divide="ignore", invalid="ignore", over="ignore"):
            proposal = sigma[active] - error / slope
        unsafe = (slope < VEGA_MIN) | ~np.isfinite(proposal) | (proposal < SIGMA_LO) | (proposal > SIGMA_HI)
        to_bisect.append(active[unsafe])
        sigma[active[~unsafe]] = proposal[~unsafe]
        active = active[~unsafe]
    # Not converged within NEWTON_MAX_ITER steps: bisection too.
    to_bisect.append(active)
    stuck = todo[np.concatenate(to_bisect)] if to_bisect else np.array([], dtype=int)

    # 4. Bisection on [SIGMA_LO, SIGMA_HI] for the rest, to a bracket of 1e-10 sigma.
    if stuck.size:
        lo = np.full(stuck.size, SIGMA_LO)
        hi = np.full(stuck.size, SIGMA_HI)
        open_ = np.arange(stuck.size)
        for step in range(1, BISECTION_MAX_ITER + 1):
            if open_.size == 0:
                break
            where = stuck[open_]
            mid = 0.5 * (lo[open_] + hi[open_])
            error = b76_price(forward[where], strike[where], tau[where], rate[where], mid, otm_call[where]) - target[where]
            high = error > 0                     # the model price is too high: sigma is too high
            hi[open_[high]] = mid[high]
            lo[open_[~high]] = mid[~high]
            narrow = (hi[open_] - lo[open_]) <= SIGMA_RTOL * hi[open_]
            iv[where[narrow]] = 0.5 * (lo[open_][narrow] + hi[open_][narrow])
            solver[where[narrow]] = SOLVER_BISECTION
            iterations[where[narrow]] = step
            open_ = open_[~narrow]
        fail[stuck[open_]] = FAIL_NO_ROOT       # never seen; kept so a non-convergence cannot pass silently
    # 5. Every answer, Newton's or bisection's: float's rounding of the price, divided by vega, must pin
    # sigma. Near the upper bound B is flat in sigma (a 30-year call at sigma 3 is within 1e-14 of DF F),
    # so a relative price match there says nothing about sigma.
    found = np.flatnonzero(np.isfinite(iv))
    with np.errstate(divide="ignore", invalid="ignore", under="ignore"):
        rounding = price_rounding(forward[found], strike[found], tau[found], rate[found], iv[found], otm_call[found])
        # An ITM price's time value, price - lower, inherits the rounding of the ITM price itself (a few
        # epsilons of ITM Black-76's two terms, which near the money are far larger than the price) and
        # of its bound. An OTM price has lower = 0 and nothing to add.
        itm = lower[found] > 0
        inherited = price_rounding(forward[found], strike[found], tau[found], rate[found], iv[found], is_call[found])
        rounding = rounding + np.where(itm, inherited + ROUNDING_ULPS * np.finfo(float).eps * lower[found], 0.0)
        slope = b76_vega(forward[found], strike[found], tau[found], rate[found], iv[found])
        pinned = rounding <= RESOLVE_SIGMA_RTOL * iv[found] * slope
    bad = found[~pinned]
    iv[bad] = np.nan
    solver[bad] = SOLVER_NONE
    iterations[bad] = 0
    fail[bad] = FAIL_UNRESOLVED
    return {"iv": iv.reshape(arrays[0].shape), "solver": solver.reshape(arrays[0].shape),
            "iterations": iterations.reshape(arrays[0].shape), "fail": fail.reshape(arrays[0].shape)}

In [ ]:
# ADR-0008 test 3: the call at 9.148603 solves back to 0.20; test 4: above DF F is IV_FAIL.
print(implied_vol(9.148603, 400, 400, 30 / 365, 0.0, True))
print(implied_vol(np.exp(-0.05 * 30 / 365) * 400 + 0.01, 400, 380, 30 / 365, 0.05, True))

## The CRR lattice and the early-exercise premium

American minus European on one lattice is the premium a mid carries for the right to exercise early. Every lattice prices at sigma no lower than 2 |r| sqrt(dt), where the up-probability is still a probability. Copied from `src/volbt/pricing/lattice.py`.

In [ ]:
import numpy as np
from scipy.special import gammaln

In [ ]:
DEAM_TREE_STEPS = 200        # ADR-0008 §3a: the de-Americanisation lattice
EXERCISE_TREE_STEPS = 500    # ADR-0003 Decision 5 and ADR-0008 §8: the early-exercise and model-mark lattice
SIGMA_FLOOR_MARGIN = 2.0     # sigma_floor = margin |r| sqrt(dt): p stays near 0.75, well inside (0, 1)
SIGMA_FLOOR_MIN = 1e-4       # and never below the Black-76 solver's own floor (iv.SIGMA_LO)

In [ ]:
def _rows(n: int, values) -> np.ndarray:
    return np.broadcast_to(np.asarray(values, dtype=float), (n,)).astype(float)

In [ ]:
def _dividends(n: int, times, amounts) -> tuple[np.ndarray, np.ndarray]:
    """(times, amounts) as (n, m) arrays; m = 0 when there are none."""
    if times is None or np.size(times) == 0:
        return np.zeros((n, 0)), np.zeros((n, 0))
    times = np.atleast_2d(np.asarray(times, dtype=float))
    amounts = np.atleast_2d(np.asarray(amounts, dtype=float))
    return (np.broadcast_to(times, (n, times.shape[1])).astype(float),
            np.broadcast_to(amounts, (n, amounts.shape[1])).astype(float))

In [ ]:
def sigma_floor(rate, tau, steps: int = DEAM_TREE_STEPS) -> np.ndarray:
    """The lowest sigma the lattice prices at: SIGMA_FLOOR_MARGIN |r| sqrt(tau / N), at least 1e-4.

    Above it the up-probability p is inside (0, 1) (module docstring, "the guard on p").
    """
    dt = np.asarray(tau, dtype=float) / steps
    return np.maximum(SIGMA_FLOOR_MIN, SIGMA_FLOOR_MARGIN * np.abs(np.asarray(rate, dtype=float)) * np.sqrt(np.maximum(dt, 0.0)))

In [ ]:
def crr(spot, strike, tau, rate, sigma, is_call, div_times=None, div_amounts=None,
        steps: int = DEAM_TREE_STEPS, american: bool = True) -> dict:
    """Price options on a CRR lattice. Returns {"price": (n,), "continuation": (n,), "up_probability": (n,)}.

    `continuation` is the value at the root of NOT exercising now: the discounted expectation of
    the next step's values. For an American put, exercise is optimal today iff
    continuation <= K - S (ADR-0003 rule, with its $0.005 tolerance in exercise.py).
    `up_probability` is the p this walk used, returned so a test can check the walk itself (M2
    round 2: a separate up_probability() function was tested instead, a copy of the formula).
    Every option needs tau > 0; sigma is raised to sigma_floor (the guard on p).
    """
    sizes = [np.size(x) for x in (spot, strike, tau, rate, sigma, is_call)]
    n = max(sizes)
    spot, strike, tau, rate, sigma = (_rows(n, x) for x in (spot, strike, tau, rate, sigma))
    sigma = np.maximum(sigma, sigma_floor(rate, tau, steps))
    sign = np.where(np.broadcast_to(np.asarray(is_call, dtype=bool), (n,)), 1.0, -1.0)
    div_t, div_d = _dividends(n, div_times, div_amounts)
    live = (div_d > 0) & (div_t > 0) & (div_t <= tau[:, None])
    div_d = np.where(live, div_d, 0.0)

    dt = tau / steps
    move = sigma * np.sqrt(dt)                   # ln u
    up = np.exp(move)
    growth = np.exp(rate * dt)
    p = (growth - 1 / up) / (up - 1 / up)
    weight_up, weight_down = p / growth, (1 - p) / growth    # the discounted step probabilities
    # The risky part of the stock, S* = S - PV of the dividends paid before expiry.
    risky = spot - (div_d * np.exp(-rate[:, None] * div_t)).sum(axis=1)

    # Layout (node, option): the first i + 1 rows are step i's nodes, one contiguous block, and every
    # step works in place in the same three buffers.
    j = np.arange(steps + 1)
    signed = sign[None, :] * risky[None, :] * np.exp((2 * j[:, None] - steps) * move[None, :])  # +-S* at expiry
    value = np.maximum(signed - sign[None, :] * strike[None, :], 0.0)
    scratch = np.empty_like(value)
    continuation = np.zeros(n)
    for i in range(steps - 1, -1, -1):
        now = value[: i + 1]
        np.multiply(value[1 : i + 2], weight_up, out=scratch[: i + 1])
        np.multiply(now, weight_down, out=now)
        np.add(now, scratch[: i + 1], out=now)          # V_i,j = e^(-r dt) [p V_i+1,j+1 + (1 - p) V_i+1,j]
        if i == 0:
            continuation = now[0].copy()
        if american:
            stock = signed[: i + 1]
            np.multiply(stock, up, out=stock)            # S*_i,j = S*_i+1,j u, with its sign
            t = i * dt
            pending = (div_d * (div_t > t[:, None]) * np.exp(-rate[:, None] * (div_t - t[:, None]))).sum(axis=1)
            # Exercising pays sign (S* + PV of the dividends still to come - K).
            np.add(stock, sign * (pending - strike), out=scratch[: i + 1])
            np.maximum(now, scratch[: i + 1], out=now)
    return {"price": value[0].copy(), "continuation": continuation, "up_probability": p}

In [ ]:
def crr_european(spot, strike, tau, rate, sigma, is_call, div_times=None, div_amounts=None,
                 steps: int = DEAM_TREE_STEPS) -> np.ndarray:
    """The European price on the same lattice, in closed form: no backward walk is needed.

    With no early exercise the root value is the discounted payoff averaged over the binomial
    distribution of the number of ups: exp(-r tau) sum_j C(N, j) p^j (1 - p)^(N - j) payoff_j.
    Equal to crr(..., american=False) to rounding (a test checks it), in O(N) instead of O(N^2).
    The same floor on sigma as crr.
    """
    n = max(np.size(x) for x in (spot, strike, tau, rate, sigma, is_call))
    spot, strike, tau, rate, sigma = (_rows(n, x) for x in (spot, strike, tau, rate, sigma))
    sigma = np.maximum(sigma, sigma_floor(rate, tau, steps))
    sign = np.where(np.broadcast_to(np.asarray(is_call, dtype=bool), (n,)), 1.0, -1.0)
    div_t, div_d = _dividends(n, div_times, div_amounts)
    live = (div_d > 0) & (div_t > 0) & (div_t <= tau[:, None])
    risky = spot - (np.where(live, div_d, 0.0) * np.exp(-rate[:, None] * div_t)).sum(axis=1)
    dt = tau / steps
    move = sigma * np.sqrt(dt)
    up = np.exp(move)
    p = (np.exp(rate * dt) - 1 / up) / (up - 1 / up)
    j = np.arange(steps + 1)
    # log of the binomial probabilities, so N = 2,000 does not overflow C(N, j).
    log_pmf = (gammaln(steps + 1) - gammaln(j + 1) - gammaln(steps - j + 1))[None, :] \
        + j[None, :] * np.log(p)[:, None] + (steps - j)[None, :] * np.log1p(-p)[:, None]
    node = risky[:, None] * np.exp((2 * j[None, :] - steps) * move[:, None])
    payoff = np.maximum(sign[:, None] * (node - strike[:, None]), 0.0)
    return np.exp(-rate * tau) * (np.exp(log_pmf) * payoff).sum(axis=1)

In [ ]:
def early_exercise_premium(spot, strike, tau, rate, sigma, is_call, div_times=None, div_amounts=None,
                           steps: int = DEAM_TREE_STEPS) -> np.ndarray:
    """American minus European on one lattice, per share: never negative (ADR-0008 §3a step 3)."""
    american = crr(spot, strike, tau, rate, sigma, is_call, div_times, div_amounts, steps)["price"]
    european = crr_european(spot, strike, tau, rate, sigma, is_call, div_times, div_amounts, steps)
    # The two sums differ in their last bits even where the premium is exactly 0.
    return np.maximum(american - european, 0.0)

In [ ]:
# ADR-0008 test 9: the ATM 30-day put at r = 5%, sigma 0.15: 6.9574 on the lattice, 6.8162 European.
american = crr(450, 450, 30 / 365, 0.05, 0.15, False, steps=500)['price'][0]
european = price(450 * np.exp(0.05 * 30 / 365), 450, 30 / 365, 0.05, 0.15, False)
print(round(float(american), 4), round(float(european), 4))
print('premium at N = 200:', early_exercise_premium(450, [430, 450, 470], 30 / 365, 0.05, 0.15, False))

## The parity forward

Each strike with a call and a put gives F_K = K + (C - P) / DF; F is the median over the 5 strikes nearest K*. Copied from `src/volbt/pricing/forward.py`.

In [ ]:
from datetime import date

import numpy as np

In [ ]:
DAYS_PER_YEAR = 365
MEDIAN_STRIKES = 5           # F is the median F_K over this many strikes nearest K*
DEAM_STRIKES = 10            # the strikes nearest K* whose call and put are de-Americanised (§3a)
MIN_PAIRS = 3                # fewer valid pairs than this: the dividend fallback
OUTLIER_REL = 0.005          # |F_K - F| / F above this: PARITY_OUTLIER

In [ ]:
def tau_years(quote_date: date, expiry: date) -> float:
    """Calendar days from the quote date to the expiry, over 365. 0 on the expiry date itself."""
    return (expiry - quote_date).days / DAYS_PER_YEAR

In [ ]:
def discount_factor(rate, tau):
    """DF = exp(-r tau), with r the continuous rate from DGS3MO (ADR-0007)."""
    return np.exp(-np.asarray(rate, dtype=float) * np.asarray(tau, dtype=float))

In [ ]:
def carry_forward(spot: float, rate: float, tau: float, div_times, div_amounts) -> float:
    """The fallback F = S exp(r tau) - sum D_i exp(r (tau - tau_i)) over dividends with 0 < tau_i <= tau.

    ADR-0008 test 7: S = 450, r = 5%, tau = 30/365, one dividend of 1.80 at 10/365 gives 450.048.
    """
    times, amounts = np.asarray(div_times, dtype=float), np.asarray(div_amounts, dtype=float)
    paid = (times > 0) & (times <= tau)
    return float(spot * np.exp(rate * tau) - (amounts[paid] * np.exp(rate * (tau - times[paid]))).sum())

In [ ]:
def strike_forwards(strikes, call_mid, put_mid, df):
    """F_K = K + (C - P) / DF at each strike."""
    return np.asarray(strikes, dtype=float) + (np.asarray(call_mid, dtype=float) - np.asarray(put_mid, dtype=float)) / df

In [ ]:
def nearest(strikes, center: float, count: int) -> np.ndarray:
    """Indices of the `count` strikes nearest `center`. A tie goes to the lower strike (stable sort)."""
    distance = np.abs(np.asarray(strikes, dtype=float) - center)
    return np.argsort(distance, kind="stable")[:count]

In [ ]:
def parity_forward(strikes, call_mid, put_mid, df: float) -> dict | None:
    """Steps 1-3 for one expiry, on strikes sorted ascending that all have a valid call and put.

    Returns {forward, k_star, used (indices of the median strikes), mad_rel}, or None when there
    are fewer than MIN_PAIRS pairs. ADR-0008 test 2: K = 443, C = 5.560, P = 6.315,
    tau = 29/365, r = 0.0521594 give DF = 0.995864 and F_K = 442.2419.
    """
    strikes = np.asarray(strikes, dtype=float)
    if strikes.size < MIN_PAIRS:
        return None
    gap = np.asarray(call_mid, dtype=float) - np.asarray(put_mid, dtype=float)
    forwards = strike_forwards(strikes, call_mid, put_mid, df)
    k_star = float(strikes[np.argmin(np.abs(gap))])   # argmin: the first (lowest) strike on a tie
    used = nearest(strikes, k_star, MEDIAN_STRIKES)
    forward = float(np.median(forwards[used]))
    mad_rel = float(np.median(np.abs(forwards[used] - forward)) / forward)
    return {"forward": forward, "k_star": k_star, "used": used, "mad_rel": mad_rel}

In [ ]:
def slope_per_strike(strikes, forwards) -> float:
    """The least-squares slope of F_K against K: dF_K/dK. NaN with fewer than 3 points.

    The American put premium grows with the strike, so raw F_K falls with K at positive rates:
    about -0.036 per strike on 2023-06-15 (ADR-0008, Revised, point 3).
    """
    strikes, forwards = np.asarray(strikes, dtype=float), np.asarray(forwards, dtype=float)
    if strikes.size < 3:
        return float("nan")
    centred = strikes - strikes.mean()
    return float((centred * (forwards - forwards.mean())).sum() / (centred**2).sum())

In [ ]:
# ADR-0008 test 2 and test 7.
df = discount_factor(0.0521594, 29 / 365)
print(round(float(df), 6), round(float(strike_forwards(443, 5.560, 6.315, df)), 4))
print(round(carry_forward(450, 0.05, 30 / 365, [10 / 365], [1.80]), 3))

## De-Americanisation, one pass by hand

A synthetic American chain (flat sigma 0.12, r = 5.22%, a $1.638 dividend tomorrow) on a 2,000-step truth lattice. The raw parity forward is biased low by the puts' premium. Each near-K* quote gets its own American IV on the lattice (A(sigma_A) = mid) and loses the premium A - E at it; parity on those European-equivalent mids recovers the forward. The premium does not depend on F, so one pass is the whole procedure (ADR-0008 §3a, as amended after the M2 review). A sigma counts only where the price pins it (round 2): deep in the money A(sigma) is flat, and such a quote is priced at its partner's American IV. Copied from `src/volbt/pricing/deam.py`; it calls the lattice, IV and Black-76 functions by module, so the next cell gives it those modules.

In [ ]:
from types import SimpleNamespace
black76 = SimpleNamespace(price=price, vega=vega)
lattice = SimpleNamespace(crr=crr, crr_european=crr_european, sigma_floor=sigma_floor,
                          DEAM_TREE_STEPS=DEAM_TREE_STEPS)
iv = SimpleNamespace(implied_vol=implied_vol, RESOLVE_SIGMA_RTOL=RESOLVE_SIGMA_RTOL,
                     FAIL_AMERICAN_BELOW_FLOOR=FAIL_AMERICAN_BELOW_FLOOR,
                     FAIL_AMERICAN_ABOVE_CAP=FAIL_AMERICAN_ABOVE_CAP,
                     FAIL_AMERICAN_NO_CONVERGENCE=FAIL_AMERICAN_NO_CONVERGENCE,
                     FAIL_AMERICAN_UNIDENTIFIED=FAIL_AMERICAN_UNIDENTIFIED, FAIL_AMERICAN_NO_MID=FAIL_AMERICAN_NO_MID,
                     FAIL_AMERICAN_EXPIRED=FAIL_AMERICAN_EXPIRED, FAIL_AMERICAN_BAD_INPUT=FAIL_AMERICAN_BAD_INPUT)

In [ ]:
import numpy as np

In [ ]:
PRICE_ATOL = 1e-8             # USD: |A(sigma) - mid| below max(this, PRICE_RTOL x mid) is converged
PRICE_RTOL = 1e-7
SIGMA_CAP = 5.0               # the bracket's top, as the Black-76 solver's
MAX_ITER = 40
# Identified also needs A to move with sigma at least half as much as E does: the premium A - E at a
# sigma off by d sigma is off by (E' - A') d sigma, so with A' >= E' / 2 an error eps in the lattice's
# American price (its discretisation, against the market's) moves the premium by at most eps.
SENSITIVITY_MIN = 0.5
STALL_ITER = 2                # bisect after this many steps that did not halve the bracket

In [ ]:
def _carry_forward(spot, rate, tau, div_times, div_amounts):
    """(S - PV(dividends before expiry)) e^(r tau): the forward the lattice itself has."""
    live = (div_amounts > 0) & (div_times > 0) & (div_times <= tau[:, None])
    pv = (np.where(live, div_amounts, 0.0) * np.exp(-rate * div_times)).sum(axis=1)
    return (spot - pv) * np.exp(rate * tau), live.any(axis=1)

In [ ]:
def american_iv(mid, spot: float, strike, tau, rate: float, is_call, div_times, div_amounts,
                steps: int = lattice.DEAM_TREE_STEPS) -> dict:
    """sigma_A with A(sigma_A) = mid, and premium A - E at it, quote by quote.

    mid, strike, tau, is_call: (n,); div_times, div_amounts: (n, m), zero-padded (lattice.py's
    rules). Returns {"iv", "premium" (NaN where it failed), "fail" (iv.FAIL_* codes),
    "walks" (backward walks spent on the quote)}.
    """
    mid, strike, tau = (np.asarray(x, dtype=float).ravel() for x in (mid, strike, tau))
    is_call = np.broadcast_to(np.asarray(is_call, dtype=bool), mid.shape).ravel()
    n = mid.size
    div_times = np.asarray(div_times, dtype=float).reshape(n, -1) if np.size(div_times) else np.zeros((n, 0))
    div_amounts = np.asarray(div_amounts, dtype=float).reshape(n, -1) if np.size(div_amounts) else np.zeros((n, 0))
    out_iv, premium = np.full(n, np.nan), np.full(n, np.nan)
    fail = np.zeros(n, dtype=np.int8)
    walks = np.zeros(n, dtype=np.int16)
    if n == 0:
        return {"iv": out_iv, "premium": premium, "fail": fail, "walks": walks}
    # 5. Inputs that cannot be priced fail at once, each with its own code (the first that applies).
    bad_input = (~np.isfinite(strike) | ~np.isfinite(tau) | (not np.isfinite(spot)) | (not np.isfinite(rate))
                 | ~np.isfinite(div_times).all(axis=1) | ~np.isfinite(div_amounts).all(axis=1))
    expired = ~bad_input & ~(tau > 0)
    no_mid = ~bad_input & ~expired & ~(np.isfinite(mid) & (mid > 0))
    fail[bad_input], fail[expired], fail[no_mid] = (iv.FAIL_AMERICAN_BAD_INPUT, iv.FAIL_AMERICAN_EXPIRED,
                                                    iv.FAIL_AMERICAN_NO_MID)
    bad = bad_input | expired | no_mid
    if bad.all():
        return {"iv": out_iv, "premium": premium, "fail": fail, "walks": walks}
    # The rest of the solve never sees the bad rows' values (a NaN would only make warnings).
    tau = np.where(bad, 1.0, tau)
    strike = np.where(bad, spot if np.isfinite(spot) else 1.0, strike)
    mid = np.where(bad, 1.0, mid)
    div_times, div_amounts = np.where(bad[:, None], 0.0, div_times), np.where(bad[:, None], 0.0, div_amounts)
    forward, has_dividend = _carry_forward(spot, rate, tau, div_times, div_amounts)
    # A call with nothing to exercise for is European: closed form, premium 0.
    walk = ~(is_call & ~has_dividend & (rate >= 0))
    floor = lattice.sigma_floor(rate, tau, steps)
    tol = np.maximum(PRICE_ATOL, PRICE_RTOL * mid)

    # 2. The start: Black-76 on the lattice's forward, less the lattice's European discretisation error.
    start = iv.implied_vol(mid, forward, strike, np.where(tau > 0, tau, 1.0), rate, is_call)["iv"]
    start = np.where(np.isfinite(start), start, 0.2)
    with np.errstate(divide="ignore", invalid="ignore", over="ignore"):
        offset = (lattice.crr_european(spot, strike, tau, rate, start, is_call, div_times, div_amounts, steps)
                  - black76.price(forward, strike, tau, rate, start, is_call))
        nudged = start - offset / black76.vega(forward, strike, tau, rate, start)
    start = np.where(np.isfinite(nudged) & (nudged > floor) & (nudged < SIGMA_CAP), nudged, start)
    sigma = np.clip(start, floor, SIGMA_CAP)

    lo, hi = floor.copy(), np.full(n, SIGMA_CAP)
    lo_seen, hi_seen = np.zeros(n, bool), np.zeros(n, bool)
    prev_sigma, prev_gap = np.full(n, np.nan), np.full(n, np.nan)
    stall = np.zeros(n, dtype=np.int16)
    a_at, e_at = np.full(n, np.nan), np.full(n, np.nan)      # A and E at the accepted sigma
    open_ = np.flatnonzero(~bad)
    for _ in range(MAX_ITER):
        if open_.size == 0:
            break
        a = open_
        price = np.empty(a.size)
        walked = walk[a]
        if walked.any():
            w = a[walked]
            price[walked] = lattice.crr(spot, strike[w], tau[w], rate, sigma[w], is_call[w], div_times[w],
                                        div_amounts[w], steps)["price"]
            walks[w] += 1
        if (~walked).any():
            c = a[~walked]
            price[~walked] = lattice.crr_european(spot, strike[c], tau[c], rate, sigma[c], is_call[c], div_times[c],
                                                  div_amounts[c], steps)
        gap = price - mid[a]
        done = np.abs(gap) <= tol[a]
        if done.any():
            d = a[done]
            european = lattice.crr_european(spot, strike[d], tau[d], rate, sigma[d], is_call[d], div_times[d],
                                            div_amounts[d], steps)
            out_iv[d] = sigma[d]
            premium[d] = np.where(walk[d], np.maximum(price[done] - european, 0.0), 0.0)
            a_at[d], e_at[d] = price[done], european
        # The bracket: A rises with sigma, so a price above the mid caps sigma, one below floors it.
        width_before = hi[a] - lo[a]
        above = gap > 0
        hi[a[above]], hi_seen[a[above]] = sigma[a[above]], True
        lo[a[~above]], lo_seen[a[~above]] = sigma[a[~above]], True
        # 4. At an edge of the bracket and still on the wrong side: there is no American IV.
        below_floor = ~done & above & (sigma[a] <= floor[a])
        above_cap = ~done & ~above & (sigma[a] >= SIGMA_CAP)
        fail[a[below_floor]] = iv.FAIL_AMERICAN_BELOW_FLOOR
        fail[a[above_cap]] = iv.FAIL_AMERICAN_ABOVE_CAP
        keep = ~done & ~below_floor & ~above_cap
        a, gap, width_before = a[keep], gap[keep], width_before[keep]
        # 3. The next sigma: secant after the first step, Newton (Black-76 vega) on it.
        with np.errstate(divide="ignore", invalid="ignore", over="ignore"):
            secant = sigma[a] - gap * (sigma[a] - prev_sigma[a]) / (gap - prev_gap[a])
            newton = sigma[a] - gap / black76.vega(forward[a], strike[a], tau[a], rate, sigma[a])
        step = np.where(np.isfinite(prev_sigma[a]) & np.isfinite(secant), secant, newton)
        # A true bracket (both ends priced) that did not halve: the secant is creeping, so bisect.
        bracketed = lo_seen[a] & hi_seen[a]
        stall[a] = np.where(bracketed & (hi[a] - lo[a] > 0.5 * width_before), stall[a] + 1, 0)
        inside = np.isfinite(step) & (step > lo[a]) & (step < hi[a]) & (stall[a] < STALL_ITER)
        # Outside the bracket: try the edge itself once (it may be where the root is not), else bisect.
        to_floor = ~inside & np.isfinite(step) & (step <= lo[a]) & ~lo_seen[a]
        to_cap = ~inside & np.isfinite(step) & (step >= hi[a]) & ~hi_seen[a]
        bisect = 0.5 * (lo[a] + hi[a])
        stall[a] = np.where(inside | to_floor | to_cap, stall[a], 0)
        prev_sigma[a], prev_gap[a] = sigma[a], gap
        sigma[a] = np.where(inside, step, np.where(to_floor, lo[a], np.where(to_cap, hi[a], bisect)))
        # A bracket closed to float precision: the sigma is as pinned as it can be.
        pinned = lo_seen[a] & hi_seen[a] & (hi[a] - lo[a] <= 1e-12 * hi[a])
        if pinned.any():
            p = a[pinned]
            american = np.where(walk[p], lattice.crr(spot, strike[p], tau[p], rate, sigma[p], is_call[p], div_times[p],
                                                     div_amounts[p], steps)["price"], np.nan)
            european = lattice.crr_european(spot, strike[p], tau[p], rate, sigma[p], is_call[p], div_times[p],
                                            div_amounts[p], steps)
            out_iv[p] = sigma[p]
            premium[p] = np.where(walk[p], np.maximum(american - european, 0.0), 0.0)
            a_at[p], e_at[p] = np.where(walk[p], american, european), european
        open_ = a[~pinned]
    fail[open_] = iv.FAIL_AMERICAN_NO_CONVERGENCE
    # 4. Identified: A(sigma (1 + d)) - A(sigma) clears the price tolerance, and is at least
    # SENSITIVITY_MIN times E(sigma (1 + d)) - E(sigma). One bump, upward, for every matched quote at
    # once: A at sigma itself is the price the solve accepted (one walk, not two, per quote).
    m = np.flatnonzero(np.isfinite(out_iv))
    if m.size:
        bumped = out_iv[m] * (1 + iv.RESOLVE_SIGMA_RTOL)
        up = np.empty(m.size)
        walked = walk[m]
        if walked.any():
            w = m[walked]
            up[walked] = lattice.crr(spot, strike[w], tau[w], rate, bumped[walked], is_call[w], div_times[w],
                                     div_amounts[w], steps)["price"]
            walks[w] += 1
        european_up = lattice.crr_european(spot, strike[m], tau[m], rate, bumped, is_call[m], div_times[m],
                                           div_amounts[m], steps)
        up[~walked] = european_up[~walked]
        spread = up - a_at[m]
        flat = ~((spread > tol[m]) & (spread >= SENSITIVITY_MIN * (european_up - e_at[m])))
        out_iv[m[flat]], premium[m[flat]] = np.nan, np.nan
        fail[m[flat]] = iv.FAIL_AMERICAN_UNIDENTIFIED
    return {"iv": out_iv, "premium": premium, "fail": fail, "walks": walks}

In [ ]:
S, r, tau, D = 442.61, 0.0521594, 29 / 365, 1.638367
K = np.arange(418.0, 468.0)
truth_F = (S - D * np.exp(-r / 365)) * np.exp(r * tau)
C = crr(S, K, tau, r, 0.12, True, [[1 / 365]], [[D]], steps=2000)['price']
P = crr(S, K, tau, r, 0.12, False, [[1 / 365]], [[D]], steps=2000)['price']
df = discount_factor(r, tau)
raw = parity_forward(K, C, P, df)
near = np.sort(nearest(K, raw['k_star'], DEAM_STRIKES))
n = near.size
mids = np.r_[C[near], P[near]]
got = american_iv(mids, S, np.r_[K[near], K[near]], np.full(2 * n, tau), r, np.r_[np.ones(n, bool), np.zeros(n, bool)],
                  np.full((2 * n, 1), 1 / 365), np.full((2 * n, 1), D))
european = mids - got['premium']
one_pass = parity_forward(K[near], european[:n], european[n:], df)
print(f"truth {truth_F:.4f}  raw {raw['forward']:.4f} ({(raw['forward'] / truth_F - 1) * 1e4:+.2f} bp)"
      f"  one pass {one_pass['forward']:.4f} ({(one_pass['forward'] / truth_F - 1) * 1e4:+.2f} bp)")
print('American IVs of the near-K* puts:', np.round(got['iv'][n:], 4))

## The smile and the surface, in total variance

Nodes (k, w = sigma^2 tau); linear in w between them, Lee-clipped beyond them, linear in tau across expiries, and along k / sqrt(tau) outside the listed tenors. Copied from `src/volbt/pricing/smile.py`.

In [ ]:
import numpy as np

In [ ]:
MIN_NODES = 5                 # ADR-0004 step 2: a smile needs at least 5 valid OTM quotes
LEE_SLOPE_MAX = 2.0           # Lee (2004): w / |k| -> at most 2 in the wings

# position_marks.mark_flags bits for the surface (ADR-0004); the engine (M3) adds its own.
SURFACE_EXTRAP_STRIKE = 1 << 0
SURFACE_CROSS_EXPIRY = 1 << 1
SURFACE_EXTRAP_TENOR = 1 << 2
SURFACE_CALENDAR_ADJUSTED = 1 << 3   # the calendar constraint moved the value, up or down: smiles crossed
SURFACE_FLAG_NAMES = ("SURFACE_EXTRAP_STRIKE", "SURFACE_CROSS_EXPIRY", "SURFACE_EXTRAP_TENOR", "SURFACE_CALENDAR_ADJUSTED")

# smile_fits.fit_flags bits.
SMILE_THIN = 1 << 0           # fewer than MIN_NODES nodes: not used by the surface
SMILE_ATM_OUTSIDE = 1 << 1    # k = 0 lies outside the nodes, so atm_iv is an extrapolation
SMILE_FORWARD_FALLBACK = 1 << 2  # the expiry's forward is the dividend fallback, not parity
SMILE_FORWARD_RAW = 1 << 3    # the forward is raw parity: too few near-K* quotes had an American IV
SMILE_FLAG_NAMES = ("SMILE_THIN", "SMILE_ATM_OUTSIDE", "SMILE_FORWARD_FALLBACK", "SMILE_FORWARD_RAW")

In [ ]:
def smile_nodes(log_moneyness, iv, tau: float) -> tuple[np.ndarray, np.ndarray]:
    """(k, w) nodes sorted by k from OTM IVs: w = iv^2 tau. NaN IVs are left out."""
    k, sigma = np.asarray(log_moneyness, dtype=float), np.asarray(iv, dtype=float)
    keep = np.isfinite(k) & np.isfinite(sigma)
    order = np.argsort(k[keep], kind="stable")
    return k[keep][order], (sigma[keep][order] ** 2) * tau

In [ ]:
def wing_slope(w_near: float, w_far: float, k_near: float, k_far: float) -> float:
    """The extrapolation slope of w per unit of |k| moving away from the nodes, clipped to [0, 2]."""
    return float(np.clip((w_far - w_near) / abs(k_far - k_near), 0.0, LEE_SLOPE_MAX))

In [ ]:
def smile_variance(k, k_nodes, w_nodes) -> tuple[np.ndarray, np.ndarray]:
    """(w at k, True where k is beyond the nodes) for one expiry: steps 2 and 3. Needs 2+ nodes.

    ADR-0004 test 5: IVs 0.22 at k = -0.05 and 0.18 at 0, tau 0.1: w(-0.025) = 0.004040.
    Test 9: IVs 0.22 at -0.10 and 0.20 at -0.05: the slope in |k| is 0.0168 and w(-0.15) = 0.00568.
    """
    k = np.asarray(k, dtype=float)
    kn, wn = np.asarray(k_nodes, dtype=float), np.asarray(w_nodes, dtype=float)
    inside = np.interp(k, kn, wn)                       # linear in w between neighbouring nodes
    left = wn[0] + wing_slope(wn[1], wn[0], kn[1], kn[0]) * (kn[0] - k)
    right = wn[-1] + wing_slope(wn[-2], wn[-1], kn[-2], kn[-1]) * (k - kn[-1])
    w = np.where(k < kn[0], left, np.where(k > kn[-1], right, inside))
    return w, (k < kn[0]) | (k > kn[-1])

In [ ]:
def smile_iv(k, tau: float, k_nodes, w_nodes) -> tuple[np.ndarray, np.ndarray]:
    """(sigma, extrapolated) at k for one expiry with time to expiry tau."""
    w, beyond = smile_variance(k, k_nodes, w_nodes)
    return np.sqrt(np.maximum(w, 0.0) / tau), beyond

In [ ]:
def _breakpoints(k_nodes, w_nodes) -> np.ndarray:
    """Where w(x) / x^2 can have an extremum: the nodes, and each linear piece's stationary point.

    On a piece w = c + b x, d/dx [(c + b x) / x^2] = -(b x + 2 c) / x^3 vanishes at x = -2c / b.
    """
    kn, wn = np.asarray(k_nodes, dtype=float), np.asarray(w_nodes, dtype=float)
    points = [kn]
    slope_in = np.diff(wn) / np.diff(kn)
    intercept = wn[:-1] - slope_in * kn[:-1]
    with np.errstate(divide="ignore", invalid="ignore"):
        stationary = -2 * intercept / slope_in
    points.append(stationary[(stationary > kn[:-1]) & (stationary < kn[1:])])
    left, right = wing_slope(wn[1], wn[0], kn[1], kn[0]), wing_slope(wn[-2], wn[-1], kn[-2], kn[-1])
    for slope, c, beyond in ((-left, wn[0] + left * kn[0], lambda x: x < kn[0]),
                             (right, wn[-1] - right * kn[-1], lambda x: x > kn[-1])):
        if slope != 0:
            x = -2 * c / slope
            if beyond(x):
                points.append(np.array([x]))
    return np.concatenate(points)

In [ ]:
def _envelope(k, x_end, k_nodes, w_nodes, take_max: bool) -> np.ndarray:
    """max (or min) over x between k and x_end of k^2 w(x) / x^2: step 5's w over a span of tenors.

    Along k / sqrt(tau) from an expiry at tau_e, the total variance at tau' is
    w(x) tau' / tau_e with x = k sqrt(tau_e / tau'), that is k^2 w(x) / x^2. So its running
    maximum (minimum) over a span of tau' is this extremum over the matching span of x. k and
    x_end have the same sign; at k = 0 the span is the single point 0.
    """
    k, x_end = np.asarray(k, dtype=float), np.asarray(x_end, dtype=float)
    lo, hi = np.minimum(k, x_end), np.maximum(k, x_end)
    points = _breakpoints(k_nodes, w_nodes)
    # Candidates per k: the two ends, and the breakpoints strictly between them.
    candidates = np.concatenate([lo[:, None], hi[:, None], np.broadcast_to(points, (k.size, points.size))], axis=1)
    within = np.concatenate([np.ones((k.size, 2), bool), (points[None, :] > lo[:, None]) & (points[None, :] < hi[:, None])],
                            axis=1)
    with np.errstate(divide="ignore", invalid="ignore"):
        ratio = smile_variance(candidates.ravel(), k_nodes, w_nodes)[0].reshape(candidates.shape) * (k[:, None] / candidates) ** 2
    ratio = np.where(within, ratio, -np.inf if take_max else np.inf)
    out = ratio.max(axis=1) if take_max else ratio.min(axis=1)
    # k = 0: w(0) tau' / tau_e at the end of the span (the ratio is 0/0 there).
    return np.where(k == 0, smile_variance(np.zeros(k.size), k_nodes, w_nodes)[0], out)

In [ ]:
def calendar_variance(w_own, anchored) -> np.ndarray:
    """The calendar-constrained w of each expiry at each k: rows are expiries in tau order, columns k.

    Quoted values beat extrapolated ones (M2 round 2, design claim 1; decision E1). A row's w(k) is
    ANCHORED where k lies inside that expiry's own nodes (a quote on each side of it).

    (a) An anchored value becomes the running maximum of the anchored values up to it, in tau
        order: two quoted smiles that cross are resolved by raising the longer one, as before.
    (b) An extrapolated value (k beyond that expiry's nodes) is clamped into [the last anchored
        floor before it, the next anchored floor after it]: a shorter expiry's extrapolated wing can
        no longer lift a longer expiry's quoted node (the first floor did: 2020-02-28, a 322-day
        node at k = -1.43 went from 0.515 to 1.242 off the 215-day wing), and it is lowered to the
        next quoted value instead of standing above it.
    (c) A final running maximum. It cannot move an anchored value, whose (a) value already bounds
        every row before it from above; it only orders extrapolated values between two anchors.

    A NaN or infinite w is a missing value (round 3, decision F7: +inf was clipped to the next floor,
    or spread to every later row): it is not an anchor and never spreads to the rows after it
    (np.fmax); the row where it is comes back NaN.
    """
    w_own = np.asarray(w_own, dtype=float)
    missing = ~np.isfinite(w_own)
    w_own = np.where(missing, np.nan, w_own)
    anchor = np.asarray(anchored, dtype=bool) & ~missing
    rows = w_own.shape[0]
    floor = np.fmax.accumulate(np.where(anchor, w_own, -np.inf), axis=0)        # (a), -inf before the first anchor
    before = np.vstack([np.full((1,) + w_own.shape[1:], -np.inf), floor[:-1]])
    # The floor at the next anchor after each row: index of the first anchored row below, +inf if none.
    index = np.where(anchor, np.arange(rows)[:, None], rows)
    following = np.minimum.accumulate(index[::-1], axis=0)[::-1]
    following = np.vstack([following[1:], np.full((1,) + w_own.shape[1:], rows)])
    after = np.where(following < rows, np.take_along_axis(np.vstack([floor, floor[-1:]]), np.minimum(following, rows), 0),
                     np.inf)
    value = np.where(anchor, floor, np.clip(w_own, before, after))              # (b); NaN stays NaN
    final = np.fmax.accumulate(value, axis=0)                                   # (c), NaN skipped
    return np.where(missing, np.nan, final)

In [ ]:
def _moved(new, old) -> np.ndarray:
    """True where the calendar constraint moved a value, beyond rounding (1e-12 of it)."""
    with np.errstate(invalid="ignore"):
        return np.abs(new - old) > 1e-12 * np.abs(old)

In [ ]:
def surface_iv(k, tau: float, expiries, calendar: bool = True) -> tuple[np.ndarray, np.ndarray]:
    """(sigma, flags) at log-moneyness k (an array) and one tau, from the usable expiries.

    `expiries` is a list of (tau_i, k_nodes, w_nodes), each with at least MIN_NODES nodes; they are
    sorted by tau_i here, and two with the same tau_i (within 1e-12) raise ValueError. Steps 2-5 and the calendar constraint of the module docstring; flags are
    SURFACE_* bits, one int per k. SURFACE_CALENDAR_ADJUSTED is set exactly where the value differs
    from steps 2-5 alone (`calendar=False`, kept to measure the constraint), up or down. At tau <= 0
    there is no implied vol: NaN, never an exception (an option on its expiry date is worth its
    intrinsic value, ADR-0003), and the NaN is returned before the equal-tau check, so equal taus
    raise only at tau > 0.
    ADR-0004 test 10: one expiry at 14/365 with IVs 0.22 at k = -0.05 and 0.18 at 0, read at
    tau = 7/365 and k = -0.025, gives 0.2091 along k/sqrt(tau) (0.2010 at the same k).
    """
    k = np.asarray(k, dtype=float)
    shape = k.shape
    k = k.ravel()
    if not expiries:
        raise ValueError("no usable expiry: the surface needs at least one smile with 5 or more nodes")
    flags = np.zeros(k.size, dtype=np.int32)
    if not (np.isfinite(tau) and tau > 0):
        return np.full(shape, np.nan), flags.reshape(shape)
    expiries = sorted(expiries, key=lambda e: float(e[0]))
    taus = np.array([e[0] for e in expiries], dtype=float)
    # Two smiles at one tau have no order, so which one the constraint raises would depend on the list's
    # order (round 3, decision F7): refused.
    same = np.flatnonzero(np.diff(taus) <= 1e-12)
    if same.size:
        raise ValueError(f"two usable expiries have the same tau ({taus[same[0]]!r} and {taus[same[0] + 1]!r}, within "
                         "1e-12 years): one smile per expiry")
    # Every expiry's own w at k (steps 2-3), and the calendar-constrained one (calendar_variance).
    own = [smile_variance(k, kn, wn) for _, kn, wn in expiries]
    w_own = np.vstack([w for w, _ in own])
    beyond = np.vstack([b for _, b in own])
    w_cal = calendar_variance(w_own, ~beyond) if calendar else w_own

    exact = np.flatnonzero(np.isclose(taus, tau, rtol=0, atol=1e-12))
    below, above = np.flatnonzero(taus < tau), np.flatnonzero(taus > tau)
    if exact.size:                                        # steps 2-3 on the own expiry
        i = exact[0]
        w, plain = w_cal[i], w_own[i]
        flags |= np.where(beyond[i], SURFACE_EXTRAP_STRIKE, 0)
    elif below.size and above.size:                       # step 4: between two expiries
        lo, hi = below[-1], above[0]
        weight = (tau - taus[lo]) / (taus[hi] - taus[lo])
        w = w_cal[lo] + (w_cal[hi] - w_cal[lo]) * weight
        plain = w_own[lo] + (w_own[hi] - w_own[lo]) * weight
        flags |= SURFACE_CROSS_EXPIRY | np.where(beyond[lo] | beyond[hi], SURFACE_EXTRAP_STRIKE, 0)
    else:                                                 # step 5: before the front or past the back
        i = below[-1] if below.size else above[0]
        t_near, k_near, w_near = expiries[i]
        x = k * np.sqrt(t_near / tau)
        plain, outside = smile_variance(x, k_near, w_near)
        plain = plain * tau / t_near
        w = plain
        if calendar and below.size:   # past the back: the running maximum since it, never below its value there
            w = np.maximum(_envelope(k, x, k_near, w_near, take_max=True) * np.where(k == 0, tau / t_near, 1.0), w_cal[i])
        elif calendar:                # before the front: the running minimum up to it, never above its value there
            w = np.minimum(_envelope(k, x, k_near, w_near, take_max=False) * np.where(k == 0, tau / t_near, 1.0), w_cal[i])
        flags |= SURFACE_EXTRAP_TENOR | np.where(outside, SURFACE_EXTRAP_STRIKE, 0)
    flags |= np.where(_moved(w, plain), SURFACE_CALENDAR_ADJUSTED, 0)
    return np.sqrt(np.maximum(w, 0.0) / tau).reshape(shape), flags.reshape(shape)

In [ ]:
# ADR-0004 tests 5, 9 and 10.
k, w = np.array([-0.05, 0.0]), np.array([0.22, 0.18]) ** 2 * 0.1
print(round(float(smile_iv(-0.025, 0.1, k, w)[0]), 4))
k, w = np.array([-0.10, -0.05]), np.array([0.22, 0.20]) ** 2 * 0.1
print(round(float(smile_iv(-0.15, 0.1, k, w)[0]), 4))
near = (14 / 365, np.array([-0.05, 0.0]), np.array([0.22, 0.18]) ** 2 * 14 / 365)
print(surface_iv(np.array([-0.025]), 7 / 365, [near]))